# VISCERA wording v3 — corrected coverage, run all

Choose **Runtime → Change runtime type → A100 GPU**, then **Run all** and approve Drive access. Upload the prepared v3 training ZIP only if it is not already on Drive. Previous datasets, models and runs remain preserved.

Train the pinned pretrained Qwen2.5-0.5B-Instruct with rank-16 LoRA for **one complete epoch at learning rate 3e-5**. This revision adds qualitative requests/defaults missing from v2. Checkpoints persist every100 steps. Re-running resumes the identical new v3 run; original v2 training is untouched.

The failed v2 audit/challenge are retained separately as **exposed regression evidence**. A new independently authored, frozen audit/challenge is packaged before v3 training. Runtime conformance and partition separation are checked before packaging. Fresh authored tests are not clinician validation or evidence of unrestricted natural-language generalization. Final activation requires all unchanged original generation gates on at least512 audit examples and the complete fresh challenge. A failed gate downloads diagnostics and exports no active weights.

The final cell downloads **viscera-wording-weights.zip** only after passing evaluation. The model creates bounded visual anatomy edit plans, not patient diagnoses, surgical mechanics or unrestricted meshes. No result is promised before testing.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
DELIVERY = Path('/content/drive/MyDrive') / 'VISCERA Wording Coverage V3 2026-10-08'
DELIVERY.mkdir(parents=True, exist_ok=True)
ARCHIVE = DELIVERY / 'viscera-wording-training-v3.zip'
EXPECTED_SHA256 = '063b1a215e95f17f00feb2435fcdc13b10fc1a6bd3222e03e5ccf411c4841eeb'
if not ARCHIVE.exists():
    print('Upload viscera-wording-training-v3.zip from the prepared delivery.')
    uploaded = files.upload()
    payload = uploaded.get('viscera-wording-training-v3.zip')
    assert payload is not None, 'The v3 training ZIP is required.'
    assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256, 'Wrong training bundle'
    with ARCHIVE.open('xb') as stream: stream.write(payload)
with ARCHIVE.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == EXPECTED_SHA256, 'Training bundle checksum mismatch'
WORK = Path('/content/viscera-wording-v3-' + EXPECTED_SHA256[:12])
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as bundle:
    names = bundle.namelist()
    assert len(names) == len(set(names)), 'Duplicate ZIP member'
    for item in bundle.infolist():
        p = Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and p.parts[0] == 'viscera-wording-training', 'Unsafe source bundle'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks are not allowed'
    bundle.extractall(WORK)
ROOT = WORK / 'viscera-wording-training'
manifest = json.loads((ROOT / 'bundle_manifest.json').read_text())
assert manifest['format'] == 'viscera-wording-training-bundle-v3-coverage'
actual = {p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*') if p.is_file()}
assert actual == set(manifest['artifact_hashes']) | {'bundle_manifest.json'}, 'Unmanifested source file'
for name, expected in manifest['artifact_hashes'].items():
    with (ROOT / name).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, f'Changed bundled file: {name}'
DATA = ROOT / 'dataset'
RUN = DELIVERY / ('run-v3-' + manifest['dataset_release_sha256'][:12])
RUN.mkdir(exist_ok=True)
print('Verified release:', manifest['dataset_release_sha256'])
print('Conformance:', json.loads((ROOT / 'preflight.json').read_text()))
print('Separate persistent run:', RUN)
print('Fresh audit/challenge frozen before training; original promotion gates apply.')

In [ ]:
"""Colab setup without a dependency on the runtime's optional ensurepip package."""

import subprocess
import sys
from pathlib import Path


def ensure_wording_environment(environment):
    environment = Path(environment).absolute()
    python = environment / "bin/python"

    def checked(command):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode:
            details = "\n".join(part.strip() for part in (result.stdout, result.stderr) if part)
            raise RuntimeError(
                f"Wording environment setup failed (exit {result.returncode}): "
                f"{command!r}\n{details}"
            )
        return result.stdout.strip()

    # Re-running this also repairs a partial venv left by a failed ensurepip step.
    # No --clear: installed packages and every Drive training checkpoint survive.
    checked([sys.executable, "-m", "venv", "--without-pip", str(environment)])
    checked([
        str(python), "-c",
        "import sys; from pathlib import Path; "
        "assert sys.prefix != sys.base_prefix, 'Expected an isolated environment'; "
        "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), "
        "'Unexpected environment prefix'",
        str(environment),
    ])
    probe = subprocess.run(
        [str(python), "-m", "pip", "--version"], capture_output=True, text=True
    )
    if probe.returncode:
        # pip's documented --python support can seed a pip-free environment.
        # This installs only into the target environment, not the Colab kernel.
        checked([
            sys.executable, "-m", "pip", "--python", str(python), "install",
            "--disable-pip-version-check", "--index-url", "https://pypi.org/simple",
            "pip==25.2",
        ])
    print("Wording environment ready:", python)
    print(checked([str(python), "-m", "pip", "--version"]))
    return python


ENV = Path('/content/viscera-wording-v3-env')
PYTHON = ensure_wording_environment(ENV)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(ROOT / 'requirements-wording-colab.txt')], check=True)
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / 'source/py') + os.pathsep + str(ROOT / 'source'), TOKENIZERS_PARALLELISM='false', PYTHONDONTWRITEBYTECODE='1')
subprocess.run([str(PYTHON), '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Select an A100 GPU runtime'; print(torch.cuda.get_device_name(0)); assert torch.cuda.get_device_properties(0).total_memory >= 35*1024**3, 'This prepared microbatch16 run requires A100 40GB or larger'; assert torch.ones(16,device='cuda').sum().item()==16"], check=True, env=ENVIRONMENT)
print('Training uses isolated Torch 2.8; no kernel restart is necessary.')

In [ ]:
# One shared runner owns training, validation, parity testing, evaluation and gated export.
# Resume never changes configuration or repeats an existing final evaluation.
OUTPUT = RUN / 'viscera-wording-weights.zip'
try:
    subprocess.run([str(PYTHON), '-u', str(ROOT / 'run_wording_v3_pipeline.py'), '--data', str(DATA), '--run', str(RUN), '--output', str(OUTPUT)], check=True, cwd=ROOT / 'source', env=ENVIRONMENT)
except subprocess.CalledProcessError:
    if (RUN / 'evaluation.json').is_file():
        files.download(str(RUN / 'evaluation.json'))
    raise
print(json.dumps(json.loads((RUN / 'pipeline_status.json').read_text()), indent=2))

In [ ]:
status = json.loads((RUN / 'pipeline_status.json').read_text())
assert status['status'] == 'complete' and status['production_exported'] is True
OUTPUT = Path(status['weights'])
with OUTPUT.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == status['sha256']
evaluation = json.loads((RUN / 'evaluation.json').read_text())
print(json.dumps({'passed': evaluation['passed'], 'gates': evaluation['gates'], 'generation_execution': evaluation.get('generation_execution')}, indent=2))
print('Validated wording weights:', OUTPUT)
print('Prior joint shape/color models are preserved. Import this separate wording lane in Studio.')
files.download(str(OUTPUT))